# Refrigerator components: 2025 trade-policy shocks

**What this notebook does.** It reads US imports of refrigerator components from `core` —
compressors (HS 841430, reviewed family) and stainless steel flat-rolled sheet (HS 721934,
candidate) — and shows where supply moved between origins and modes around the 2025
trade-policy shocks, and what that did to the transport emissions of delivering a tonne to
the US.

**Why refrigeration.** GE Appliances (Haier) manufactures refrigerators using steel cabinets
and compressors, with supply chains running from Chinese manufacturers to US ports. The 2025
tariffs created a component backlog. Competitors (manufacturing in the US and Sweden) expected
a demand surge. Unlike servers (heading 8471, excluded from the reciprocal tariffs), neither
heading 8414 (compressors) nor heading 7219 (steel sheet) was excluded, so both stayed
exposed.

**The results at a glance** are in Section 8.

**How it’s organised.** Each section computes with the engine on data read by
`analysis/src/shockt_analysis/core.py`. The windows reuse the Case 1 tariff timeline.

**Reporting boundary.** Census trade data is aggregate and does not prove a GE Appliances or
Haier supplier, shipment, or destination. Do not report these results as company-specific
emissions without separate evidence.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells.

## 1. Setup

Two product groups: compressors (reviewed family) and steel sheet (candidate, review required).
The windows are the Case 1 tariff timeline (the reciprocal tariffs apply to these components
the same as to servers, but without the exclusion). Named origins are China, Taiwan, Mexico,
Thailand, Vietnam and South Korea — the six from the handover package.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import polars as pl
from shockt_engine import (
    Window,
    load_leg_factors,
    mix_decomposition,
    select_factors,
    window_summary,
)

from shockt_analysis import cases, core
from shockt_analysis.settings import REPO_ROOT, get_settings

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)

CASE = "refrigeration"
CASE_DIR = REPO_ROOT / "inputs" / "cases" / CASE
NOTEBOOK = "analysis/refrigeration/refrigeration_tariffs.ipynb"

COMPRESSORS = ["841430"]
STEEL_CANDIDATE = ["721934"]
ALL_REFRIG = COMPRESSORS + STEEL_CANDIDATE
NAMED_ORIGINS = ["CHN", "TWN", "MEX", "THA", "VNM", "KOR"]

windows = cases.load_windows(CASE_DIR / "shock_windows.yaml")
WINDOW_COLORS = {"baseline": "#d9e5d6", "anticipation": "#f3dfb6", "response": "#f0c7c0"}
[(w.name, w.first, w.last) for w in windows]

Read everything the notebook needs in one connection.

In [ ]:
with core.connect(get_settings()) as conn:
    supply = core.supply(conn, ALL_REFRIG)
    node_weights = core.node_weights(conn, ALL_REFRIG)
    observed_sea = core.observed_sea(conn)
    distances = {seed: core.distances(conn, seed) for seed in ("sea", "air", "road", "rail")}
    crossing_plants = core.crossing_plants(conn)
    factor_rows = core.factors(conn)
    loads = core.current_loads(
        conn,
        [
            "census_hs",
            "census_porths",
            "census_statehs",
            "bts_transborder",
            "climatetrace_voyages",
            "bts_t100_segments",
            "uk_ghg_conversion_factors",
            "epa_ghg_transport_factors",
            "epa_ghg_gwp",
        ],
    )

leg_factors = select_factors(factor_rows, load_leg_factors())
context = cases.LaneContext(
    weights=node_weights,
    observed_sea=observed_sea,
    distances=distances,
    factors={mode: f.wtw_kg_co2e_per_tkm for mode, f in leg_factors.items()},
    crossing_plants=crossing_plants,
)
print(f"Supply rows: {supply.height:,}; months {supply['month'].min()} to {supply['month'].max()}")
pl.DataFrame(
    [
        {
            "mode": mode,
            "factor": f.variant,
            "source": f.source,
            "kg CO2e per tonne-km": f.wtw_kg_co2e_per_tkm,
        }
        for mode, f in leg_factors.items()
    ]
)

In [ ]:
loads

## 2. Monthly supply

Tonnes of refrigerator components delivered to the US each month, by origin, with the windows
shaded. First all components, then compressors and steel separately.

In [ ]:
def shade_windows(ax):
    for w in windows:
        ax.axvspan(
            w.first,
            date(w.last.year + (w.last.month == 12), w.last.month % 12 + 1, 1),
            color=WINDOW_COLORS[w.name],
            alpha=0.5,
            lw=0,
            label=w.name,
        )


def plot_by_origin(codes, named, title):
    monthly = cases.monthly_supply(supply, codes, named)
    pivot = (
        monthly.group_by("month", "origin_group")
        .agg(pl.col("weight_t").sum())
        .pivot(on="origin_group", index="month", values="weight_t")
        .sort("month")
        .fill_null(0)
    )
    order = [o for o in named + [cases.OTHER] if o in pivot.columns]
    fig, ax = plt.subplots(figsize=(11, 4))
    shade_windows(ax)
    ax.stackplot(pivot["month"].to_list(), *[pivot[o].to_list() for o in order], labels=order)
    ax.set_title(title)
    ax.set_ylabel("tonnes per month")
    ax.legend(loc="upper left", ncols=4, fontsize=8)
    plt.show()
    return monthly


refrig_monthly = plot_by_origin(
    ALL_REFRIG, NAMED_ORIGINS, "Refrigerator components delivered to the US, by origin"
)

In [ ]:
compressor_monthly = plot_by_origin(
    COMPRESSORS, NAMED_ORIGINS, "Compressors (841430) delivered to the US, by origin"
)
steel_monthly = plot_by_origin(
    STEEL_CANDIDATE, NAMED_ORIGINS,
    "Stainless steel sheet candidate (721934) delivered to the US, by origin",
)

Mode shares within each origin.

In [ ]:
mode_shares = cases.window_shares(refrig_monthly, windows, ["origin_group", "mode"])
by_origin = mode_shares.group_by("window", "origin_group").agg(
    pl.col("weight_t").sum().alias("origin_t")
)
(
    mode_shares.join(by_origin, on=["window", "origin_group"])
    .with_columns((pl.col("weight_t") / pl.col("origin_t")).alias("share_within_origin"))
    .pivot(on="window", index=["origin_group", "mode"], values="share_within_origin")
    .select("origin_group", "mode", *[w.name for w in windows])
    .sort("origin_group", "mode")
)

## 3. Origin shares and changes

Each origin's share of refrigerator component tonnage per window, and the change from baseline
to response. China is expected to be the dominant origin for compressors.

In [ ]:
origin_shares = cases.window_shares(refrig_monthly, windows, ["origin_group"])
shares_table = origin_shares.pivot(
    on="window", index="origin_group", values="share"
).select("origin_group", *[w.name for w in windows])
changes = cases.share_changes(origin_shares, "baseline", "response", "origin_group")
shares_table.with_columns(
    pl.col("origin_group")
    .replace_strict(changes, default=None)
    .alias("change_baseline_to_response")
)

## 4. By HS family: compressors vs steel

Do the two product families move alike? If China's share drops for compressors but not for
steel (or vice versa), the two may serve different supply chains, which weakens the case for
combining them. The steel family (721934) is a candidate — this comparison is part of the
evidence needed to validate or reject it.

In [ ]:
families = {
    "compressors (841430)": COMPRESSORS,
    "steel candidate (721934)": STEEL_CANDIDATE,
}
family_results = []
for label, codes in families.items():
    fm = cases.monthly_supply(supply, codes, NAMED_ORIGINS)
    ws = cases.window_shares(fm, windows, ["origin_group"])
    ch = cases.share_changes(ws, "baseline", "response", "origin_group")
    for origin, change_val in ch.items():
        family_results.append({
            "family": label,
            "origin_group": origin,
            "change_pp": change_val * 100,
        })

family_df = pl.DataFrame(family_results)
family_df.pivot(
    on="family", index="origin_group", values="change_pp"
).sort("origin_group")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (label, codes) in zip(axes, families.items(), strict=True):
    fm = cases.monthly_supply(supply, codes, NAMED_ORIGINS)
    ws = cases.window_shares(fm, windows, ["origin_group"])
    ch = cases.share_changes(ws, "baseline", "response", "origin_group")
    origins_sorted = sorted(ch.keys(), key=lambda o: ch[o])
    vals = [ch[o] * 100 for o in origins_sorted]
    ax.barh(
        origins_sorted, vals,
        color=["#a2412d" if v < 0 else "#2d5b4f" for v in vals],
    )
    ax.set_xlabel("Share change, percentage points")
    ax.set_title(label)
    ax.axvline(0, color="black", lw=0.5)
plt.suptitle("Origin share changes, baseline to response", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Delivery emissions per tonne, by lane

A lane is an origin and a delivery mode. Its emissions per tonne are the sum over its legs of
distance × factor, with each leg’s distance averaged over the airports, ports and states the
lane uses. Sea distances are observed where they exist, and `searoute` distances otherwise.

In [ ]:
refrig_lanes, refrig_legs = cases.lane_intensities(supply, ALL_REFRIG, windows, context)
(
    refrig_lanes.filter(pl.col("iso3").is_in(NAMED_ORIGINS))
    .pivot(on="window", index=["iso3", "mode"], values="kg_co2e_per_t")
    .select("iso3", "mode", *[w.name for w in windows])
    .sort("iso3", "mode")
)

In [ ]:
print("Lanes with tonnage but no measurable legs (left out of the mix):")
missing = refrig_lanes.filter(pl.col("kg_co2e_per_t").is_null())
display(missing)
missing_t = missing["weight_t"].sum()
total_t = refrig_lanes["weight_t"].sum()
print(
    f"\n{missing_t:,.0f} t of {total_t:,.0f} t total ({missing_t / total_t:.1%}) have no "
    f"measurable legs."
)

## 6. The supply mix

The emissions per tonne of all US refrigerator component supply in a month: each origin
lane’s share of the month’s tonnes × the lane’s intensity.

In [ ]:
mix = {
    estimate: cases.mix_by_month(
        supply, ALL_REFRIG, windows, refrig_lanes, tonnes=column
    )
    for estimate, column in [
        ("central", "weight_t"),
        ("mexico low", "weight_t_low"),
        ("mexico high", "weight_t_high"),
    ]
}
fig, ax = plt.subplots(figsize=(11, 4))
shade_windows(ax)
central = mix["central"]
ax.plot(
    central["month"].to_list(),
    central["kg_co2e_per_t"].to_list(),
    marker="o",
    color="#2d5b4f",
    label="central",
)
ax.fill_between(
    central["month"].to_list(),
    mix["mexico high"]["kg_co2e_per_t"].to_list(),
    mix["mexico low"]["kg_co2e_per_t"].to_list(),
    color="#2d5b4f",
    alpha=0.15,
    label="Mexico tonnage range",
)
ax.set_ylabel("kg CO2e per tonne delivered")
ax.set_title("Emissions per tonne of US refrigerator component supply")
ax.legend(loc="upper left", fontsize=8)
plt.show()

values = dict(zip(central["month"].to_list(), central["kg_co2e_per_t"].to_list(), strict=True))
weights = dict(zip(central["month"].to_list(), central["t"].to_list(), strict=True))
mix_summary = window_summary(values, windows, weights)
pl.DataFrame(
    [
        {
            "window": name,
            "mean_kg_per_t": s.mean,
            "low": s.low,
            "high": s.high,
            "months": s.months,
            "change_from_baseline": s.mean / mix_summary["baseline"].mean - 1
            if name != "baseline"
            else 0.0,
        }
        for name, s in mix_summary.items()
    ]
)

In [ ]:
print(f"Measured share of tonnage (central): "
      f"{central['measured_share'].mean():.1%} of monthly supply has a measured lane.")

## 7. What moved the mix

The Shapley decomposition: was the change in emissions per tonne driven by the origin mix
(which countries supply how much), the mode within each origin (air vs vessel), or the route
legs (sea distance, inland distribution)?

In [ ]:
refrig_cells = {
    w.name: cases.lane_mix_cells(supply, ALL_REFRIG, w, refrig_legs) for w in windows
}
refrig_split = mix_decomposition(
    refrig_cells["baseline"], refrig_cells["response"], ["origin", "mode"]
)
fig, ax = plt.subplots(figsize=(9, 4))
cases.plot_waterfall(
    ax,
    refrig_split,
    {
        "origin": "origin mix",
        "mode": "mode within origin",
        "international": "legs to the US",
        "inland": "legs in the US",
    },
    "What moved the emissions per tonne of US refrigerator component supply, baseline to response",
    window_label="response",
)
plt.show()
change = refrig_split.window - refrig_split.baseline
if abs(change) > 0:
    print(
        f"Of the {change:+,.0f} kg CO2e per tonne change, the origin mix accounts for "
        f"{refrig_split.effects['origin']:+,.0f} kg "
        f"({refrig_split.effects['origin'] / change:.0%}), "
        f"the mode within each origin {refrig_split.effects['mode']:+,.0f}, the legs to the US "
        f"{refrig_split.effects['international']:+,.0f} and the legs inside it "
        f"{refrig_split.effects['inland']:+,.0f}."
    )
else:
    print("No measurable change between baseline and response.")

## 8. Results at a glance

Three charts: **emissions per tonne** by period, **tonnes per month by origin**, and **what
moved the mix**.

In [ ]:
last_window = windows[-1].last
after = Window(
    "after",
    date(last_window.year + last_window.month // 12, last_window.month % 12 + 1, 1),
    supply["month"].max(),
)
after_lanes, _ = cases.lane_intensities(supply, ALL_REFRIG, [after], context)
periods = [*windows, after]
period_lanes = pl.concat([refrig_lanes, after_lanes])

monthly_mix = cases.mix_by_month(supply, ALL_REFRIG, periods, period_lanes)
months = monthly_mix["month"].to_list()
summary = window_summary(
    dict(zip(months, monthly_mix["kg_co2e_per_t"].to_list(), strict=True)),
    periods,
    dict(zip(months, monthly_mix["t"].to_list(), strict=True)),
)
period_labels = [
    f"{w.first:%b %Y}\u2013{w.last:%b %Y}\n({w.name})" for w in windows
] + [f"{after.first:%b %Y}\u2013{after.last:%b %Y}\n(after)"]
means = [summary[w.name].mean for w in periods]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    period_labels,
    means,
    color=[WINDOW_COLORS.get(w.name, "#e0e0e0") for w in periods],
    edgecolor="#2d5b4f",
)
ax.set_ylabel("kg CO2e per tonne delivered")
ax.set_title("US refrigerator component supply: emissions per tonne by period")
ax.tick_params(axis="x", labelsize=8)
plt.show()

base_mean = summary["baseline"].mean
resp_mean = summary["response"].mean
print(
    f"Emissions per tonne: {base_mean:.1f} kg in baseline, {resp_mean:.1f} kg in response "
    f"({resp_mean / base_mean - 1:+.0%})."
)

In [ ]:
origin_tonnes = (
    cases.with_window(refrig_monthly, windows)
    .drop_nulls("window")
    .group_by("window", "origin_group")
    .agg(pl.col("weight_t").sum())
    .with_columns(
        (
            pl.col("weight_t")
            / pl.col("window").replace_strict(
                {
                    w.name: (w.last.year - w.first.year) * 12 + w.last.month - w.first.month + 1
                    for w in windows
                }
            )
        ).alias("t_per_month")
    )
    .pivot(on="window", index="origin_group", values="t_per_month")
    .with_columns((pl.col("response") / pl.col("baseline")).alias("growth"))
    .sort("baseline", descending=True)
)

fig, ax = plt.subplots(figsize=(9, 4))
groups_ = origin_tonnes["origin_group"].to_list()
x = range(len(groups_))
ax.bar(
    [i - 0.2 for i in x],
    origin_tonnes["baseline"].to_list(),
    0.4,
    label="baseline",
    color=WINDOW_COLORS["baseline"],
    edgecolor="#2d5b4f",
)
ax.bar(
    [i + 0.2 for i in x],
    origin_tonnes["response"].to_list(),
    0.4,
    label="response",
    color=WINDOW_COLORS["response"],
    edgecolor="#a2412d",
)
for i, g in zip(x, origin_tonnes["growth"].to_list(), strict=True):
    if g is not None:
        ax.annotate(
            f"\u00d7{g:.2f}",
            (i + 0.2, origin_tonnes["response"][i]),
            ha="center",
            va="bottom",
            fontsize=8,
        )
ax.set_xticks(list(x), groups_)
ax.set_ylabel("tonnes per month")
ax.set_title("Refrigerator components delivered to the US: tonnes per month by origin")
ax.legend(fontsize=8)
plt.show()

## 9. Findings

In [ ]:
chn_change = changes.get("CHN", 0) * 100
twn_change = changes.get("TWN", 0) * 100
mex_change = changes.get("MEX", 0) * 100
tha_change = changes.get("THA", 0) * 100
vnm_change = changes.get("VNM", 0) * 100
kor_change = changes.get("KOR", 0) * 100

base, resp = mix_summary["baseline"], mix_summary["response"]

missing_lanes = refrig_lanes.filter(pl.col("kg_co2e_per_t").is_null())
missing_origins = sorted(missing_lanes["iso3"].unique().to_list())

findings = [
    f"China's share of US refrigerator component tonnage changed by {chn_change:+.1f} "
    f"percentage points from the baseline to the response window.",
    f"Taiwan changed by {twn_change:+.1f} pp, Mexico by {mex_change:+.1f} pp, "
    f"Thailand by {tha_change:+.1f} pp, Vietnam by {vnm_change:+.1f} pp, "
    f"South Korea by {kor_change:+.1f} pp.",
    f"The emissions per tonne of US refrigerator component supply averaged {base.mean:.1f} kg "
    f"CO2e in the baseline and {resp.mean:.1f} kg in the response window "
    f"({resp.mean / base.mean - 1:+.0%}).",
]

if abs(change) > 0:
    findings.append(
        f"The origin mix accounts for {refrig_split.effects['origin']:+,.0f} kg of the "
        f"{change:+,.0f} kg change; the mode mix {refrig_split.effects['mode']:+,.0f} kg; "
        f"legs to the US {refrig_split.effects['international']:+,.0f} kg; "
        f"legs inside the US {refrig_split.effects['inland']:+,.0f} kg."
    )

if missing_origins:
    findings.append(
        f"Vessel lanes for {', '.join(missing_origins)} have no measurable emissions; "
        f"{missing_lanes['weight_t'].sum():,.0f} t of "
        f"{refrig_lanes['weight_t'].sum():,.0f} t total "
        f"({missing_lanes['weight_t'].sum() / refrig_lanes['weight_t'].sum():.1%}) are excluded "
        f"from the mix."
    )

for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")

**Steel candidate note.** The steel sheet family (721934) is classified as REVIEW_REQUIRED.
The per-family comparison in Section 4 shows whether compressors and steel move alike. If they
do not, the steel candidate may serve a different supply chain and should be dropped from the
combined analysis. Do not report steel-candidate results as confirmed refrigerator component
flows without separate classification evidence.

**What it can’t say.** Census data is aggregate: it does not identify GE Appliances, Haier or
any other company. Monthly data cannot separate tariff effects from seasonal or demand-driven
changes in the same months.